In [11]:
pip install torch pandas numpy scikit-learn sentence-transformers lime

In [12]:
# =============================================================================
# ROBUST FAKE NEWS DETECTOR FOR MISOVAC DATASET
# =============================================================================

import os, re, random, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sentence_transformers import SentenceTransformer
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from lime.lime_text import LimeTextExplainer

warnings.filterwarnings('ignore')

# =============================================================================
# CONFIGURATION
# =============================================================================
class Config:
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    SEED = 42
    CSV_PATH = "MiSoVac.csv"          # your dataset file name

    HIDDEN_DIM = 64
    DROPOUT = 0.3
    LEARNING_RATE = 1e-3
    WEIGHT_DECAY = 1e-3
    BATCH_SIZE = 16
    EPOCHS = 30
    NOISE_LEVEL = 0.05                # data augmentation strength

config = Config()
print(f"Running on: {config.DEVICE}")

random.seed(config.SEED)
np.random.seed(config.SEED)
torch.manual_seed(config.SEED)

# =============================================================================
# 1. DATA LOADING & CLEANING
# =============================================================================
def clean_text(text):
    """Aggressively remove artifacts to force semantic learning."""
    text = str(text)
    # Remove URLs
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)
    # Remove handles and hashtags (content remains, tags go)
    text = re.sub(r'\@\w+', '', text)
    text = re.sub(r'\#\w+', '', text)
    # Remove common dataset artifacts
    artifacts = ["RT", "Edit:", "Link in description", "Subscribe", "TL;DR", "r/COVID19"]
    for art in artifacts:
        text = text.replace(art, "")
    # Remove non‑ASCII characters
    text = text.encode('ascii', 'ignore').decode('ascii')
    return text.strip()

def load_data(path):
    print("📂 Loading and cleaning data...")
    if not os.path.exists(path):
        raise FileNotFoundError(f"Could not find {path}")

    df = pd.read_csv(path)

    # Convert boolean labels to 'real' / 'fake' strings
    # True → real (factual), False → fake (misinformation)
    df['label'] = df['label'].map({True: 'real', False: 'fake'})

    # Rename platform column if needed (your file uses 'platform_label')
    if 'platform_label' in df.columns:
        df.rename(columns={'platform_label': 'platform'}, inplace=True)

    # Clean the text
    df['text'] = df['text'].apply(clean_text)

    # Remove rows that became too short after cleaning
    df = df[df['text'].str.len() > 10]

    # Deduplicate (crucial for generalization)
    before = len(df)
    df = df.drop_duplicates(subset=['text', 'label'])
    print(f"✓ Cleaned & deduplicated: {before} → {len(df)} unique samples")

    return df

# =============================================================================
# 2. EMBEDDING MANAGER
# =============================================================================
class EmbeddingManager:
    def __init__(self):
        self.model = SentenceTransformer("all-MiniLM-L6-v2")

    def get_embeddings(self, texts):
        return self.model.encode(texts, convert_to_numpy=True, show_progress_bar=False)

# =============================================================================
# 3. DATASET AND MODEL (Linear DANN)
# =============================================================================
class EmbDataset(Dataset):
    def __init__(self, X, y, d, noise_level=0.0):
        self.X = X
        self.y = y
        self.d = d
        self.noise_level = noise_level

    def __len__(self):
        return len(self.y)

    def __getitem__(self, i):
        emb = self.X[i]
        if self.noise_level > 0:
            noise = np.random.normal(0, self.noise_level, emb.shape)
            emb = emb + noise
        return (torch.tensor(emb).float(),
                torch.tensor(self.y[i]).long(),
                torch.tensor(self.d[i]).long())

class LinearDANN(nn.Module):
    """Simplified DANN for small datasets."""
    def __init__(self, input_dim, hidden_dim, n_labels, n_domains):
        super().__init__()
        self.feature_extractor = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(config.DROPOUT)
        )
        self.label_classifier = nn.Linear(hidden_dim, n_labels)
        self.domain_classifier = nn.Linear(hidden_dim, n_domains)

    def forward(self, x, grl_coeff=1.0):
        features = self.feature_extractor(x)
        reversed_features = features * -grl_coeff + features.detach() * (1 + grl_coeff)
        return self.label_classifier(features), self.domain_classifier(reversed_features)

# =============================================================================
# 4. TRAINING AND EVALUATION
# =============================================================================
def train_and_test():
    # Load and prepare data
    df = load_data(config.CSV_PATH)

    # Generate embeddings
    emb_manager = EmbeddingManager()
    print("🔢 Generating embeddings (this may take a minute)...")
    embeddings = emb_manager.get_embeddings(df['text'].tolist())

    # Encode labels and domains
    le_label = LabelEncoder()
    le_domain = LabelEncoder()
    y = le_label.fit_transform(df['label'])
    d = le_domain.fit_transform(df['platform'].fillna('unknown'))

    # Train‑test split (stratified by label)
    X_train, X_test, y_train, y_test, d_train, d_test = train_test_split(
        embeddings, y, d, test_size=0.2, random_state=config.SEED, stratify=y
    )

    # Datasets (noise only on training set)
    train_ds = EmbDataset(X_train, y_train, d_train, noise_level=config.NOISE_LEVEL)
    test_ds  = EmbDataset(X_test, y_test, d_test, noise_level=0.0)

    train_loader = DataLoader(train_ds, batch_size=config.BATCH_SIZE, shuffle=True)
    test_loader  = DataLoader(test_ds,  batch_size=config.BATCH_SIZE)

    # Model
    model = LinearDANN(
        input_dim=384,
        hidden_dim=config.HIDDEN_DIM,
        n_labels=len(le_label.classes_),
        n_domains=len(le_domain.classes_)
    ).to(config.DEVICE)

    optimizer = torch.optim.AdamW(model.parameters(),
                                  lr=config.LEARNING_RATE,
                                  weight_decay=config.WEIGHT_DECAY)
    criterion = nn.CrossEntropyLoss()

    print("🏋️  Training...")
    for epoch in range(config.EPOCHS):
        model.train()
        for x, label, domain in train_loader:
            x, label, domain = x.to(config.DEVICE), label.to(config.DEVICE), domain.to(config.DEVICE)

            # GRL coefficient: increases from -1 to 1 over epochs
            p = epoch / config.EPOCHS
            grl_coeff = 2.0 / (1.0 + np.exp(-10 * p)) - 1.0

            optimizer.zero_grad()
            l_out, d_out = model(x, grl_coeff)
            loss = criterion(l_out, label) + 0.1 * criterion(d_out, domain)
            loss.backward()
            optimizer.step()

    # Evaluation
    model.eval()
    preds = []
    with torch.no_grad():
        for x, _, _ in test_loader:
            out, _ = model(x.to(config.DEVICE))
            preds.extend(torch.argmax(out, 1).cpu().numpy())

    print("\n📊 Test Set Performance:")
    print(classification_report(y_test, preds, target_names=le_label.classes_))

    return model, emb_manager, le_label

# =============================================================================
# 5. INTERACTIVE TESTING (with LIME explanation)
# =============================================================================
if __name__ == "__main__":
    model, emb_manager, le_label = train_and_test()
    explainer = LimeTextExplainer(class_names=le_label.classes_)

    print("\n" + "="*60)
    print("🧪  ROBUST FAKE NEWS TESTER")
    print("   (Model adapted to MiSoVac dataset)")
    print("="*60)

    while True:
        user_input = input("\n📝 Enter news text (or 'q' to quit): ")
        if user_input.lower() in ['q', 'exit']:
            break

        # Clean input exactly like training data
        clean_input = clean_text(user_input)

        # Predict
        emb = emb_manager.get_embeddings([clean_input])[0]
        model.eval()
        with torch.no_grad():
            x = torch.tensor(emb).float().unsqueeze(0).to(config.DEVICE)
            out, _ = model(x, 0.0)
            probs = torch.softmax(out, 1).cpu().numpy()[0]

        pred_idx = np.argmax(probs)
        pred_label = le_label.classes_[pred_idx]
        confidence = probs[pred_idx]

        print(f"\n🔍 Prediction: {pred_label.upper()}  (confidence: {confidence:.2%})")

        # LIME explanation
        try:
            def predict_proba(texts):
                embeddings = emb_manager.get_embeddings(texts)
                x_tensor = torch.tensor(embeddings).float().to(config.DEVICE)
                with torch.no_grad():
                    out, _ = model(x_tensor, 0.0)
                    probs = torch.softmax(out, 1).cpu().numpy()
                return probs

            exp = explainer.explain_instance(clean_input, predict_proba, num_features=6)
            print("\n💡 Why? (top words):")
            for feature, weight in exp.as_list():
                # weight positive → contributes to 'real', negative → contributes to 'fake'
                # (assuming le_label.classes_[0] is 'real')
                direction = "REAL" if weight > 0 else "FAKE"
                print(f"  {feature:<20}  {direction} ({weight:+.4f})")
        except Exception as e:
            print("  (Explanation could not be generated for this short input)")

        print("-" * 50)

Running on: cuda
📂 Loading and cleaning data...
✓ Cleaned & deduplicated: 2022 → 1915 unique samples


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


🔢 Generating embeddings (this may take a minute)...
🏋️  Training...

📊 Test Set Performance:
              precision    recall  f1-score   support

        fake       0.84      0.86      0.85       204
        real       0.83      0.82      0.82       179

    accuracy                           0.84       383
   macro avg       0.84      0.84      0.84       383
weighted avg       0.84      0.84      0.84       383


🧪  ROBUST FAKE NEWS TESTER
   (Model adapted to MiSoVac dataset)

📝 Enter news text (or 'q' to quit): The FDA has authorized the Pfizer-BioNTech COVID-19 vaccine for emergency use in adolescents aged 12 to 15 years.

🔍 Prediction: REAL  (confidence: 78.15%)

💡 Why? (top words):
  Pfizer                REAL (+0.1279)
  19                    REAL (+0.0831)
  FDA                   REAL (+0.0609)
  COVID                 FAKE (-0.0544)
  vaccine               FAKE (-0.0500)
  use                   REAL (+0.0411)
--------------------------------------------------

📝 Enter news t

Example Texts
✅ Real (True) examples
Example text	Expected label
The FDA has authorized the Pfizer-BioNTech COVID-19 vaccine for emergency use in adolescents aged 12 to 15 years.	real
Clinical trials show that the Moderna vaccine has an efficacy of 94.1% in preventing symptomatic COVID-19.	real
Health experts recommend that even people who have recovered from COVID-19 should get vaccinated to boost their immunity.	real
More than 2 billion doses of COVID-19 vaccines have been administered worldwide as of June 2021.	real
❌ Fake / Misinformation examples
Example text	Expected label
Bill Gates is using the COVID-19 vaccine to implant microchips into people so the government can track their every move.	fake
The coronavirus vaccine will permanently alter your DNA and turn you into a genetically modified organism.	fake
A doctor in Italy has revealed that the COVID-19 vaccine actually contains a live strain of the virus and has killed hundreds of people.	fake
Dr. Fauci admitted that the pandemic was a hoax created to force vaccines on the entire population.	fake
🧪 Borderline / Tricky examples (model should still classify reasonably)
Example text	Expected label (most likely)
I heard the vaccine might cause infertility in women, but I haven’t seen any scientific proof yet.	fake (unsubstantiated claim)
The vaccine is safe and effective, but some people may experience mild side effects like fatigue or arm pain.	real
They say the virus is not dangerous, and the real danger is the vaccine itself.	fake
